# Imports and data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from google.colab import files
# from google.colab import drive
import photoproductionmodel as pm
np.set_printoptions(precision=10, suppress=True)

real_rho_data = pm.read_csv("selected_rho_real_dataset.csv")
# virtual_rho_sig_data = pm.read_csv(r"C:\Users\malco\Downloads\darkphysicsJun15\full_rho_virtual_sig_dataset.csv")
# virtual_rho_dsig_data = pm.read_csv(r"C:\Users\malco\Downloads\darkphysicsJun15\full_rho_virtual_dsig_dataset.csv")
omega_data = pm.read_csv("selected_omega_dataset.csv")
phi_data = pm.read_csv("selected_phi_dataset.csv")

# Constants and initial values

In [2]:
g_pi_gam_omega = 1.83 # Updated
g_eta_gam_omega = 0.35

g_pi_gam_rho = 0.57
g_eta_gam_rho = 1.22

g_pi_gam_phi = 0.138
g_eta_gam_phi = 0.704

f_omega = 0.201 # GeV
f_phi = 0.228 # GeV
f_rho = 0.220 # GeV

# Pomeron fixed parameters
beta_PNN = 1.87 # 1/GeV
alpha_prime_P_0 = 0.25 # 1/GeV^2

# f2 Reggeon fixed parameters (Ewerz et al 2013)
g_f2Rpp = 11.04
M0 = 1 # GeV
alpha_prime_R_plus = 0.9 # 1/GeV^-2

alpha_EM = 1/137.035999139

mp = 0.9382720813 # GeV
me = 0.511e-3 # GeV
m_omega = 0.78266 # GeV Updated
m_rho = 0.77526  # GeV
m_phi = 1.019461 # GeV
m_pi = 0.1349770 # GeV
m_eta = 0.547862 # GeV

Gamma_omega = 8.68e-3 # GeV Updated
Gamma_phi = 4.249e-3 # GeV
Gamma_rho = 0.1474 # GeV

units = 389.3793656 # Conversion from 1/GeV^2 to mu_barn

default_params_23 = np.zeros(23, dtype='float')
default_params_23[0] = 14.0 # g_pi_NN
default_params_23[1] = 4.10 # g_eta_NN
default_params_23[2] = 0.74 # Lambda_pi_NN
default_params_23[3] = 0.70 # Lambda_eta_NN
default_params_23[4] = 0.80 # Lambda_pi_gam_omega
default_params_23[5] = 0.80 # Lambda_eta_gam_omega
default_params_23[6] = 0.50 # Lambda_pi_gam_phi
default_params_23[7] = 0.60 # Lambda_eta_gam_phi
default_params_23[8] = 7.40 # b_Pom_omega_omega
default_params_23[9] = 2.00 # b_Pom_phi_phi
default_params_23[10] = 7.40 # B_omega (form factor slope)
default_params_23[11] = 3.10 # B_phi (form factor slope)
default_params_23[12] = 1.091 # alpha_Pom (Pomeron trajectory)
default_params_23[13] = 0.2 # a_Pom_omega_omega
default_params_23[14] = 0.9 # a_Pom_phi_phi
default_params_23[15] = 0.80 # Lambda_pi_gam_rho
default_params_23[16] = 0.80 # Lambda_eta_gam_rho
default_params_23[17] = 7.40 # b_Pom_rho_rho
default_params_23[18] = 0.2 # a_Pom_rho_rho
default_params_23[19] = 7.4 # B_rho (form factor slope)
default_params_23[20] = 1 # a_f2R_rho_rho
default_params_23[21] = 1 # b_f2R_rho_rho
default_params_23[22] = 0.55 # B_rho (form factor slope)

default_params_20 = np.zeros(20, dtype='float')
default_params_20[0] = 14.0 # g_pi_NN
default_params_20[1] = 4.10 # g_eta_NN
default_params_20[2] = 0.74 # Lambda_pi_NN
default_params_20[3] = 0.70 # Lambda_eta_NN
default_params_20[4] = 0.80 # Lambda_pi_gam_omega
default_params_20[5] = 0.80 # Lambda_eta_gam_omega
default_params_20[6] = 0.50 # Lambda_pi_gam_phi
default_params_20[7] = 0.60 # Lambda_eta_gam_phi
default_params_20[8] = 7.40 # b_Pom_omega_omega
default_params_20[9] = 2.00 # b_Pom_phi_phi
default_params_20[10] = 7.40 # B_omega (form factor slope)
default_params_20[11] = 3.10 # B_phi (form factor slope)
default_params_20[12] = 1.091 # alpha_Pom (Pomeron trajectory)
default_params_20[13] = 0.2 # a_Pom_omega_omega
default_params_20[14] = 0.9 # a_Pom_phi_phi
default_params_20[15] = 0.80 # Lambda_pi_gam_rho
default_params_20[16] = 0.80 # Lambda_eta_gam_rho
default_params_20[17] = 7.40 # b_Pom_rho_rho
default_params_20[18] = 0.2 # a_Pom_rho_rho
default_params_20[19] = 7.4 # B_rho (form factor slope)

#Prior

In [3]:
cent_values_20 = np.array([13, 4, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0, 0, 0, 0, 0, 0, 0, 0.8, 0.8, 0, 0, 0])
prior_errors_20 = np.array([1, 1, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 1e5, 1e5, 1e5, 1e5, 1e5, 1e5, 1e5, 0.2, 0.2, 1e5, 1e5, 1e5])

cent_values_23 = np.array([13, 4, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0, 0, 0, 0, 0, 0, 0, 0.8, 0.8, 0, 0, 0, 0 ,0 ,0])
prior_errors_23 = np.array([1, 1, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 1e5, 1e5, 1e5, 1e5, 1e5, 1e5, 1e5, 0.2, 0.2, 1e5, 1e5, 1e5, 1e5, 1e5, 1e5])

def chi_sq_priors(x):
  # Require Eq. (A8) from Ben Othman 2311.00043
  # a_PVV > 0 and b_PVV + 2 * mV^2 * a_PVV > 0
  # for all V = omega, phi, rho
  a_Pom_omega_omega, b_Pom_omega_omega = x[13], x[8]
  a_Pom_rho_rho, b_Pom_rho_rho = x[18], x[17]
  a_Pom_phi_phi, b_Pom_phi_phi = x[14], x[9]

  # If any inequalities not satisfied, return infinity
  if (a_Pom_omega_omega < 0) or (a_Pom_rho_rho < 0) or (a_Pom_phi_phi < 0):
      return np.inf
  elif (2*m_omega**2 * a_Pom_omega_omega + a_Pom_omega_omega < 0):
      return np.inf
  elif (2*m_phi**2 * a_Pom_phi_phi + b_Pom_phi_phi < 0):
      return np.inf
  elif (2*m_rho**2 * a_Pom_rho_rho + b_Pom_rho_rho < 0):
      return np.inf

  # Else impose Gaussian prior on parameters
  else:
      if len(x) == 20:
        return np.sum((x - cent_values_20)**2/prior_errors_20**2)
      elif len(x) == 23:
        return np.sum((x - cent_values_23)**2/prior_errors_23**2)

# Likelyhood

In [4]:
def chi_sq_rho_real(x):
    W = real_rho_data['W']
    t = real_rho_data['t']
    obs = real_rho_data['dsigdt (mu barn/GeV^2)']
    err = real_rho_data['dsigdt error (mu barn/GeV^2)']


    # Calculate total error including a 10% systematic error
    sys_err = 0.1 * obs
    err_tot = np.sqrt( err**2 + sys_err**2 )

    model = pm.dsig_dt_rho(W,t,x)

    return np.sum((model - obs)**2 / err_tot** 2)

def chi_sq_omega_data(x):

    W = omega_data['W']
    t = omega_data['t']
    obs = omega_data['dsigdt (mu barn/GeV^2)']
    err = omega_data['dsigdt error (mu barn/GeV^2)']

    # Calculate total error including a 10% systematic error
    sys_err = 0.1 * obs
    err_tot = np.sqrt( err**2 + sys_err**2 )

    model = pm.dsig_dt_omega(W,t,x)

    return np.sum((model - obs)**2/err_tot**2)

def chi_sq_phi_data(x):

    W = phi_data['W']
    t = phi_data['t']
    obs = phi_data['dsigdt (mu barn/GeV^2)']
    err = phi_data['dsigdt error (mu barn/GeV^2)']

    # Calculate total error including a 10% systematic error
    sys_err = 0.1 * obs
    err_tot = np.sqrt( err**2 + sys_err**2 )

    model = pm.dsig_dt_phi(W,t,x)

    return np.sum((model - obs)**2/err_tot**2)

def chi_sq_gamma_proton(x):

    # See Eq. (A9) from Ben Othman 2311.00043

    # Data from H1 expt
    obs_H1, stat_error_H1, sys_error_H1 = 165, 2, 11 # mu barn
    tot_error_H1 = np.sqrt(stat_error_H1**2 + sys_error_H1**2)

    # H1 CM energy
    E_CM = 200 # GeV
    s_tot = E_CM**2

    # Model prediction for H1 energy
    model_H1 = pm.sig_gamma_proton_tot(s_tot,x)

    # Data from Zeus expt
    obs_Zeus, stat_error_Zeus, sys_error_Zeus = 174, 1, 13 # mu barn
    tot_error_Zeus = np.sqrt(stat_error_Zeus**2 + sys_error_Zeus**2)

    # Zeus CM energy
    E_CM = 209 # GeV
    s_tot = E_CM**2

    # Model prediction for Zeus energy
    model_Zeus = pm.sig_gamma_proton_tot(s_tot,x)

    return (model_H1 - obs_H1)**2 / tot_error_H1**2 + (model_Zeus - obs_Zeus)**2 / tot_error_Zeus**2

def chi_sq_rho_virtual_sig(x):
    W = virtual_rho_sig_data['W']
    Q2 = virtual_rho_sig_data['Q2']
    s_tot = virtual_rho_sig_data['s_tot']
    obs = virtual_rho_sig_data['sig (mu barn)']
    err = virtual_rho_sig_data['sig error (mu barn)']


    # Calculate total error including a 10% systematic error
    sys_err = 0.1 * obs
    err_tot = np.sqrt( err**2 + sys_err**2 )

    model = pm.sig_virtual_rho(s_tot,W,Q2,x)

    return np.sum((model - obs)**2 / err_tot** 2)

def chi_sq_rho_virtual_dsig(x):
    W = virtual_rho_dsig_data['W']
    t = virtual_rho_dsig_data['t']
    Q2 = virtual_rho_dsig_data['Q2']
    s_tot = virtual_rho_dsig_data['s_tot']
    obs = virtual_rho_dsig_data['dsigdt (mu barn/GeV^2)']
    err = virtual_rho_dsig_data['dsigdt error (mu barn/GeV^2)']

    # Calculate total error including a 10% systematic error
    sys_err = 0.1 * obs
    err_tot = np.sqrt( err**2 + sys_err**2 )

    model = pm.dsig_dt_virtual_rho(s_tot,W,Q2,t,x)

    return np.sum((model - obs)**2 / err_tot** 2)

# Posterior

In [5]:
def chi_sq_tot(x):
    return chi_sq_rho_real(x) + chi_sq_omega_data(x) + chi_sq_phi_data(x) + chi_sq_priors(x) + chi_sq_gamma_proton(x)

def logP(x): # prior is encoded :D
    return -0.5 * chi_sq_tot(x)

# MCMC

In [6]:
def mcmc(N,x0,P,cov):
  """
  Args:
    N: int num of iteration
    x-:arr [initial value of params]
    P: posterior function

  Returns:
    MCMC chain
  """
  # init
  d = np.size(x0)
  x_samples = np.zeros((N, d)) # rows: iterations , cols: params
  x_samples[0] = x0
  counter = 0

  # algo
  for i in range(N - 1):
    x_old = x_samples[i]

    ## proposal ##
    x_new = np.random.multivariate_normal(x_old,cov)

    ## accept/reject for log ##
    dlogP = logP(x_new) - logP(x_old)

    if (dlogP >= 0) or (np.log(np.random.rand()) < dlogP):
      x_samples[i + 1] = x_new
      counter+= 1
    else:
      x_samples[i + 1] = x_old

  return x_samples,counter

# Save chain function

In [7]:
def save_chain(chain,name):
    np.save(name,chain)
    files.download(name)

# Convergence Testing Function

In [15]:
## Convergence testing ##
def convergence_test(chain,N):
    x1 = np.mean(chain[:int(N/2)],axis=0)
    x2 = np.mean(chain[int(N/2):],axis=0)
    stdx = np.std(chain,axis=0)
    epsilon = 0.1
    convergence_bool = abs(x1 - x2) < (epsilon * stdx)

    return all(convergence_bool)

# Manual Convergence Test

In [9]:
# convergence details printed ##
x1 = np.mean(chain[:int(len(chain)/2)],axis=0)
# print(chain[:int(len(chain)/2)][-1])
# print(chain[int(len(chain)/2):][-1])
x2 = np.mean(chain[int(len(chain)/2):],axis=0)
stdx = np.std(chain,axis=0)
epsilon = 0.1
convergence_bool = abs(x1 - x2) < (epsilon * stdx)
print(convergence_bool)
c_b_flat = convergence_bool.flat
for i in range(len(convergence_bool)):
    print(str(i) + ' convergence: ' + str(c_b_flat[i]))
    print(str(i) + ' mean first half: ' + str(x1[i]))
    print(str(i) + ' mean second half: ' + str(x2[i]))
    print(str(i) + ' difference: ' + str(abs(x1[i]-x2[i])))
    print(str(i) + ' epsilon_std: ' + str(stdx[i] * epsilon))
    print()

NameError: name 'chain' is not defined

#Plotting final params function

In [16]:
def plot_real_photoproduction(x,real_rho_data):

    real_rho_data = pd.DataFrame(real_rho_data)
    fig, axes = plt.subplots(3, 3, figsize=(14,10))

    ## plotting empirical data ##
    sources = ['ballam','derrick','Eisenburg','Aid','Wu','breitweg00','breitweg98']
    for j in range(len(sources)):
        ax = axes.flat[j]
        i = sources[j]
        src_indices = real_rho_data['source'] == i
        src_df = real_rho_data[src_indices] ## creating a DF with just the highlighted source data
        unique_Ws = np.unique(src_df['W'])
        num_unique_W = len(unique_Ws) ## all real photoproduction data is going to be plotted as dsig vs t, since t is more varying than W

        for k in range(num_unique_W):
            unique_W_indices = src_df['W']  == unique_Ws[k]
            plt_df = src_df[unique_W_indices]

            ax.errorbar(
                plt_df['t'],
                plt_df['dsigdt (mu barn/GeV^2)'],
                plt_df['dsigdt error (mu barn/GeV^2)'],
                label='W = ' + str(unique_Ws[k]),
                fmt = 'o'
                )

            ax.plot(
                np.linspace(-1,0,50),
                pm.dsig_dt_rho(
                    unique_Ws[k],
                    np.linspace(-1,0,50),
                    x),
                label = 'model'
                ) # solving the model 50 times won't be viable for integrated cross sections

            ax.legend()
            ax.set_ylabel('dsigdt (mu barn/GeV^2)')
            ax.set_xlabel('t (units)')
            ax.set_title(i + ' data')
    plt.tight_layout()
    plt.savefig('final params.png')
    files.download('final params.png')
    plt.show()

# manual plotting final params

In [ ]:
final_params = np.mean(chain,axis=0)
# plot_real_photoproduction(final_params,real_rho_data)

print(final_params)

# Execution functions

In [17]:
## initialize ##
d=23 ##SPECIFY##
cov_initial = 1e-6 * np.eye(d)
N = 100000

In [18]:
## trace plotting ##
## trace plotting ##
def plot_traces(chain,acceptance,traceplotname):
    fig, axes = plt.subplots(5, 5, figsize=(14,10), sharex=True)
    axes = axes.ravel()
    for i in range(d):
        axes[i].plot(chain[:, i], lw=0.7)
        axes[i].set_title(i)
    plt.tight_layout()
    convergence = convergence_test(chain,len(chain))
    plt.suptitle('ACCEPTANCE: ' + str(acceptance) + '  CON: ' + str(convergence))
    plt.savefig(traceplotname)
    files.download(traceplotname)

## burn ##
def burn_in_run(x,cov,N):
    chain, counter = mcmc(N,x,logP,cov)
    acceptance = counter/N
    x0 = chain[-1]
    return chain, x0, acceptance

## cov adapt ##
def adapt_cov_run(x,chain,N,scaling_factor,half):

    if half == True:
      cov = scaling_factor * np.cov(chain[int(len(chain)/2):],rowvar=False)
    else:
      cov = scaling_factor * np.cov(chain,rowvar=False)
    chain, counter = mcmc(N,x,logP,cov)
    acceptance = counter/N
    x0 = chain[-1]

    return chain, x0, acceptance

# execution, 20 params


In [ ]:
d=20
traceplot_name = "trace.png"

## burn in ##
cov_initial = 1e-6 * np.eye(d)
cov = cov_initial
N = 100000
chain, counter = mcmc(N, default_params_20,logP,cov)
x0 = chain[-1]
acceptance = counter/N
print( "final params: " + str(x0))

## trace plotting ##
fig, axes = plt.subplots(5, 4, figsize=(14,10), sharex=True)
axes = axes.ravel()
print(type(chain))
for i in range(20):
    axes[i].plot(chain[:, i], lw=0.7)
    axes[i].set_title(i)
plt.tight_layout()
plt.suptitle('ACCEPTANCE: ' + str(acceptance))
plt.savefig(traceplot_name)
files.download(traceplot_name)

In [ ]:
## burn in again ##
N = 100000
chain, counter = mcmc(N, x0,logP,cov)
x0 = chain[-1]
acceptance = counter/N
print( "final params: " + str(x0))

## trace plotting ##
fig, axes = plt.subplots(5, 4, figsize=(14,10), sharex=True)
axes = axes.ravel()
print(type(chain))
for i in range(20):
    axes[i].plot(chain[:, i], lw=0.7)
    axes[i].set_title(i)
plt.tight_layout()
plt.suptitle('ACCEPTANCE: ' + str(acceptance))
plt.savefig(traceplot_name)
files.download(traceplot_name)

In [ ]:
## cov adapt ##
cov = 0.01 * np.cov(chain[int(N/2):],rowvar=False)

## final run attempt 1 ##
N = 100000
chain, counter = mcmc(N, x0,logP,cov)
x0 = chain[-1]
acceptance = counter/N
print( "final params: " + str(x0))

## trace plotting ##
fig, axes = plt.subplots(5, 4, figsize=(14,10), sharex=True)
axes = axes.ravel()
print(type(chain))
for i in range(20):
    axes[i].plot(chain[:, i], lw=0.7)
    axes[i].set_title(i)
plt.tight_layout()
plt.suptitle('ACCEPTANCE: ' + str(acceptance))
plt.savefig(traceplot_name)
files.download(traceplot_name)

In [ ]:
## cov adapt ## computed cov should be smaller due to prev chain, but needs to be generally bigger due to prev high acceptance that we want to tank... hence 0.1 scaling
cov = 0.1 * np.cov(chain[int(N/2):],rowvar=False)

## final run attempt 1 ##
N = 100000
chain, counter = mcmc(N, x0,logP,cov)
x0 = chain[-1]
acceptance = counter/N
print( "final params: " + str(x0))

## trace plotting ##
fig, axes = plt.subplots(5, 4, figsize=(14,10), sharex=True)
axes = axes.ravel()
print(type(chain))
for i in range(d):
    axes[i].plot(chain[:, i], lw=0.7)
    axes[i].set_title(i)
plt.tight_layout()
plt.suptitle('ACCEPTANCE: ' + str(acceptance))
plt.savefig(traceplot_name)
files.download(traceplot_name)

In [ ]:
## cov adapt ## computed cov should have small step sizes due to prev chain, acceptance is good so we dont need to downscale cov, which would further increase acceptance.
cov = np.cov(chain[int(N/2):],rowvar=False)

## final run attempt 1 ##
N = 100000
chain, counter = mcmc(N, x0,logP,cov)
x0 = chain[-1]
acceptance = counter/N
print( "final params: " + str(x0))

## trace plotting ##
fig, axes = plt.subplots(5, 4, figsize=(14,10), sharex=True)
axes = axes.ravel()
print(type(chain))
for i in range(d):
    axes[i].plot(chain[:, i], lw=0.7)
    axes[i].set_title(i)
plt.tight_layout()
plt.suptitle('ACCEPTANCE: ' + str(acceptance))
plt.savefig(traceplot_name)
files.download(traceplot_name)

In [ ]:
## cov adapt ## last acceptance tiny so we will recalc cov based on entire chain, then diminish it significantly
cov = 0.01 * np.cov(chain,rowvar=False)

## final run attempt 1 ##
N = 100000
chain, counter = mcmc(N, x0,logP,cov)
x0 = chain[-1]
acceptance = counter/N
print( "final params: " + str(x0))

## trace plotting ##
fig, axes = plt.subplots(5, 4, figsize=(14,10), sharex=True)
axes = axes.ravel()
print(type(chain))
for i in range(d):
    axes[i].plot(chain[:, i], lw=0.7)
    axes[i].set_title(i)
plt.tight_layout()
plt.suptitle('ACCEPTANCE: ' + str(acceptance))
plt.savefig(traceplot_name)
files.download(traceplot_name)

In [ ]:
## cov adapt ## just updating cov w/ last half and applying same scaling factor
cov = 0.01 * np.cov(chain[int(N/2):],rowvar=False)

## final run attempt 1 ##
N = 100000
chain, counter = mcmc(N, x0,logP,cov)
x0 = chain[-1]
acceptance = counter/N
print( "final params: " + str(x0))

## trace plotting ##
fig, axes = plt.subplots(5, 4, figsize=(14,10), sharex=True)
axes = axes.ravel()
print(type(chain))
for i in range(d):
    axes[i].plot(chain[:, i], lw=0.7)
    axes[i].set_title(i)
plt.tight_layout()
plt.suptitle('ACCEPTANCE: ' + str(acceptance))
plt.savefig(traceplot_name)
files.download(traceplot_name)

In [ ]:
chain = np.load("chain.npy")
x0 = np.load("x0.npy")

In [ ]:
## cov adapt ## just updating cov w/ last half and applying larger scaling factor to bring down acceptance
N = 100000
cov = 0.1 * np.cov(chain[int(N/2):],rowvar=False)

## final run attempt 1 ##
chain, counter = mcmc(N, x0,logP,cov)
x0 = chain[-1]
acceptance = counter/N
print( "final params: " + str(x0))



In [ ]:
d = 20
traceplot_name = 'trace.png'
## trace plotting ##
fig, axes = plt.subplots(5, 4, figsize=(14,10), sharex=True)
axes = axes.ravel()
for i in range(d):
    axes[i].plot(chain[:, i], lw=0.7)
    axes[i].set_title(i)
plt.tight_layout()
plt.suptitle('ACCEPTANCE: ' + str(acceptance))
plt.savefig(traceplot_name)
files.download(traceplot_name)

In [ ]:
d = 20
traceplot_name = 'trace.png'
N = 100000
## cov adapt ## updating cov w/ full chain, scaling
cov = 0.05 * np.cov(chain,rowvar=False)

## final run attempt 1 ##
chain, counter = mcmc(N, x0,logP,cov)
x0 = chain[-1]
acceptance = counter/N
print( "final params: " + str(x0))

## trace plotting ##
fig, axes = plt.subplots(5, 4, figsize=(14,10), sharex=True)
axes = axes.ravel()
for i in range(d):
    axes[i].plot(chain[:, i], lw=0.7)
    axes[i].set_title(i)
plt.tight_layout()
plt.suptitle('ACCEPTANCE: ' + str(acceptance))
plt.savefig(traceplot_name)
files.download(traceplot_name)


#execution, 23 params

In [ ]:
## initialize ##
d=23
cov_initial = 1e-6 * np.eye(d)
N = 100000

In [ ]:
## burn in ##
for i in range(2):
    if i == 0:
        chain, new_params, acceptance = burn_in_run(default_params_23,cov_initial,100000)
        plot_traces(chain,acceptance, str(i+n) + '_burn_in.png')
        save_chain(chain,str(i+n) + '_burnin_chain.npy')
    else:
        chain, new_params, acceptance = burn_in_run(new_params,cov_initial,100000)
        plot_traces(chain,acceptance, str(i+n) + '_burn_in.png')
        save_chain(chain,str(i+n) + '_burnin_chain.npy')


In [ ]:
## burn in ##
n = 41 # numbered number
for i in range(2):
    chain, new_params, acceptance = burn_in_run(new_params,cov_initial,100000)
    plot_traces(chain,acceptance, str(i+n) + '_burn_in.png')
    save_chain(chain,str(i+n) + '_burnin_chain.npy')

In [ ]:
## adapted runs ##
n = 43
for i in range(1):
    chain, new_params, acceptance = adapt_cov_run(new_params,chain,100000,0.01,half = True)
    plot_traces(chain,acceptance,str(i+n) + '_adapted_cov_run.png')
    save_chain(chain,str(i+n) + '_adapted_chain.npy')

In [ ]:
chain, new_params, acceptance = adapt_cov_run(new_params,chain,1000000,0.01,half = True)
plot_traces(chain,acceptance,str(43) + '_adapted_cov_run.png')
save_chain(chain,str(43) + '_adapted_chain.npy')

# Load chain, params

In [12]:
chainname = '42_adapted_chain.npy' ###INPUT###
chain = np.load(chainname)
new_params = chain[-1]
# plot_traces(chain,0,'test.png')

#Load n chains

In [ ]:
chain0name = '40_adapted_chain.npy'
chain0 = np.load(chain0name)
chain1name = '41_adapted_chain.npy' ###INPUT###
chain1 = np.load(chain1name)

chain = np.append(chain0,chain1,axis=0)

# chain2name = '41_adapted_chain.npy' ###INPUT###
# chain2 = np.load(chain2name)
# chain3name = '42_adapted_chain.npy' ###INPUT###
# chain3 = np.load(chain3name)
# chain = np.append(np.append(np.append(chain1,chain2,axis=0),chain3,axis=0),chain0,axis=0)

print(np.shape(chain))
plot_traces(chain,0,'test.png')

# NOT USABLE Plotting method, Notes

In [ ]:
# questions
# DEF x0 = [14,4,0.8,0.8,0.8,0.8,0.8,0.8,7.4,2,7.4,3.1,1.091,0.3,0.9,0.8,0.8,3,0.2,5]
# figgy01.png has inital value of param set to 7 instead of 4

# tasks
# automate this so u can just press enter and it goes in the background and automatically downloads figures

# arr = np.array([112,112,112,321,321,321,745,745,745])
# unique_index = np.unique(arr,return_index=True)
# print(unique_index)
def plot_data(x,only_data):
  virtual_rho_sig_data=pd.DataFrame(virtual_rho_sig_data)

  for i in ['Joos','Cassel','Adloff','aid','chekanov','aaron']:

      src_index = virtual_rho_sig_data['source'] == i

      df = virtual_rho_sig_data[src_index]

      W = virtual_rho_sig_data['W'][src_index]
      Q2 = virtual_rho_sig_data['Q2'][src_index]
      s_tot = virtual_rho_sig_data['s_tot'][src_index]
      obs = virtual_rho_sig_data['sig (mu barn)'][src_index]
      err = virtual_rho_sig_data['sig error (mu barn)'][src_index]

      if (i == 'Joos' or i == 'Cassel'):
        unique_W, unique_index = np.unique(W,return_index=True)
        for k in unique_W:
          W_index = df['W'] == k
          plt.errorbar(Q2[W_index],obs[W_index],err[W_index],fmt='o',label='W = ' + str(k))
          if only_data:
            plt.plot(Q2[W_index],pm.sig_virtual_rho(s_tot[W_index],W[W_index],Q2[W_index],default_params),label='W = ' + str(k))
          plt.legend()
          plt.title(str(i))
          plt.xlabel('Q2')
          plt.ylabel('sig (mu barns)')

      elif (i == 'Adloff'):

        by_Q2_index = df['W'] != 75
        df1 = df[by_Q2_index]
        by_W_index = df['W'] == 75
        df2 = df[by_W_index]

        W = df['W'][by_Q2_index]
        Q2 = df['Q2'][by_Q2_index]
        s_tot = df['s_tot'][by_Q2_index]
        obs = df['sig (mu barn)'][by_Q2_index]
        err = df['sig error (mu barn)'][by_Q2_index]

        unique_Q2, unique_index = np.unique(Q2,return_index=True)
        for k in unique_Q2:
          Q2_index = df1['Q2'] == k
          plt.errorbar(Q2[Q2_index],obs[Q2_index],err[Q2_index],fmt='o',label='Q2 = ' + str(k))
          plt.plot(Q2[Q2_index],pm.sig_virtual_rho(s_tot[Q2_index],W[Q2_index],Q2[Q2_index],default_params),label='Q2 = ' + str(k))
          plt.legend()
          plt.title(str(i))
          plt.xlabel('Q2')
          plt.ylabel('sig (mu barns)')



      plt.savefig(str(i) + "_plot.png")
      plt.show()
      # files.download(str(i) + "_plot.png")

Phases for final inference

phase 1 - burn in runs, n = 10k, do not update cov, 1 time
phase 2 - n = 10k, update the cov based on the last half of the chain, 10 times
phase 3 - final run (longer)
phase 4 - comparing means in first and last half of the final run